In [ ]:
"""
SCRIPT 04 - REVISED

Menghitung jarak Haversine dari setiap desa prioritas
ke FKTP terdekat.

Menggunakan:
    Desa_Revised.csv
    FKTP.csv

Kemudian memisahkan output menjadi:

    Maluku
    Maluku Utara

Output:

    01_Hasil_Euclidean_Maluku_Revised.csv
    01_Hasil_Euclidean_Maluku_Revised.xlsx

    01_Hasil_Euclidean_Maluku_Utara_Revised.csv
    01_Hasil_Euclidean_Maluku_Utara_Revised.xlsx
"""


from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder

OUTPUT_DIR = PROJECT_ROOT / "Output"

OUTPUT_DIR.mkdir(
    exist_ok=True
)


# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------

DESA_FILE = OUTPUT_DIR / "Desa_Revised.csv"

FKTP_FILE = OUTPUT_DIR / "FKTP.csv"


# ------------------------------------------------------------
# OUTPUT MALUKU
# ------------------------------------------------------------

MALUKU_CSV = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Maluku_Revised.csv"
)

MALUKU_XLSX = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Maluku_Revised.xlsx"
)


# ------------------------------------------------------------
# OUTPUT MALUKU UTARA
# ------------------------------------------------------------

MALUKU_UTARA_CSV = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Maluku_Utara_Revised.csv"
)

MALUKU_UTARA_XLSX = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Maluku_Utara_Revised.xlsx"
)


# ============================================================
# KONSTANTA
# ============================================================

R = 6378137.0


# ============================================================
# HAVERSINE FUNCTION
# ============================================================

def haversine(
    lon1,
    lat1,
    lon2,
    lat2
):

    """
    Menghitung jarak Haversine.

    Output:
        Jarak dalam meter
    """

    lon1 = np.radians(
        float(lon1)
    )

    lat1 = np.radians(
        float(lat1)
    )

    lon2 = np.radians(
        np.asarray(
            lon2,
            dtype=float
        )
    )

    lat2 = np.radians(
        np.asarray(
            lat2,
            dtype=float
        )
    )

    dlon = lon2 - lon1

    dlat = lat2 - lat1

    a = (

        np.sin(
            dlat / 2
        )
        ** 2

    ) + (

        np.cos(lat1)

        *

        np.cos(lat2)

        *

        np.sin(
            dlon / 2
        )
        ** 2

    )

    return (

        2

        *

        R

        *

        np.arcsin(

            np.sqrt(a)

        )

    )


# ============================================================
# MAIN
# ============================================================

def main():

    # ========================================================
    # 1. READ DATA
    # ========================================================

    print("=" * 70)

    print(
        "MEMBACA DATA DESA REVISED"
    )

    print("=" * 70)

    desa = pd.read_csv(
        DESA_FILE,
        dtype=str
    )

    fktp = pd.read_csv(
        FKTP_FILE,
        dtype=str
    )


    print(
        f"Jumlah Desa : {len(desa)}"
    )

    print(
        f"Jumlah FKTP : {len(fktp)}"
    )


    # ========================================================
    # 2. CONVERT COORDINATES
    # ========================================================

    desa["Longitude"] = pd.to_numeric(
        desa["Longitude"],
        errors="coerce"
    )

    desa["Latitude"] = pd.to_numeric(
        desa["Latitude"],
        errors="coerce"
    )

    fktp["Longitude"] = pd.to_numeric(
        fktp["Longitude"],
        errors="coerce"
    )

    fktp["Latitude"] = pd.to_numeric(
        fktp["Latitude"],
        errors="coerce"
    )


    # ========================================================
    # 3. REMOVE INVALID COORDINATES
    # ========================================================

    desa_before = len(desa)

    fktp_before = len(fktp)


    desa = desa.dropna(
        subset=[
            "Longitude",
            "Latitude"
        ]
    )


    fktp = fktp.dropna(
        subset=[
            "Longitude",
            "Latitude"
        ]
    )


    print()

    print(
        f"Desa tanpa koordinat valid "
        f"yang dibuang: "
        f"{desa_before - len(desa)}"
    )

    print(
        f"FKTP tanpa koordinat valid "
        f"yang dibuang: "
        f"{fktp_before - len(fktp)}"
    )


    if len(desa) == 0:

        raise ValueError(
            "Tidak ada desa dengan koordinat valid."
        )


    if len(fktp) == 0:

        raise ValueError(
            "Tidak ada FKTP dengan koordinat valid."
        )


    # ========================================================
    # 4. PREPARE FKTP COORDINATES
    # ========================================================

    fktp_lon = fktp[
        "Longitude"
    ].to_numpy(
        dtype=float
    )


    fktp_lat = fktp[
        "Latitude"
    ].to_numpy(
        dtype=float
    )


    # ========================================================
    # 5. CALCULATE NEAREST FKTP
    # ========================================================

    rows = []


    print()

    print(
        "Menghitung FKTP terdekat..."
    )


    for d in desa.itertuples(
        index=False
    ):

        jarak = haversine(

            d.Longitude,

            d.Latitude,

            fktp_lon,

            fktp_lat

        )


        i = int(

            np.argmin(

                jarak

            )

        )


        f = fktp.iloc[i]


        rows.append(

            {

                "KDDESA":

                    d.KDDESA,


                "NMDESA2":

                    d.NMDESA2,


                "NMKEC":

                    d.NMKEC,


                "NMDATI2":

                    d.NMDATI2,


                "NMPROP":

                    d.NMPROP,


                "Longitude_Desa":

                    d.Longitude,


                "Latitude_Desa":

                    d.Latitude,


                "kode_ppk":

                    f["kode_ppk"],


                "nmppk":

                    f["nmppk"],


                "Longitude_FKTP":

                    f["Longitude"],


                "Latitude_FKTP":

                    f["Latitude"],


                "Jarak_Meter":

                    round(

                        float(

                            jarak[i]

                        ),

                        2

                    ),


                "Jarak_KM":

                    round(

                        float(

                            jarak[i]

                        )

                        /

                        1000,

                        3

                    )

            }

        )


    # ========================================================
    # 6. CREATE RESULT DATAFRAME
    # ========================================================

    hasil = pd.DataFrame(
        rows
    )


    hasil = hasil.sort_values(

        [

            "NMPROP",

            "NMDATI2",

            "NMKEC",

            "NMDESA2"

        ]

    )


    hasil = hasil.reset_index(
        drop=True
    )


    # ========================================================
    # 7. SEPARATE PROVINCES
    # ========================================================

    maluku = hasil[
        hasil["NMPROP"]
        .astype(str)
        .str.strip()
        .str.upper()
        == "MALUKU"
    ].copy()


    maluku_utara = hasil[
        hasil["NMPROP"]
        .astype(str)
        .str.strip()
        .str.upper()
        == "MALUKU UTARA"
    ].copy()


    # ========================================================
    # 8. SAVE MALUKU
    # ========================================================

    maluku.to_csv(

        MALUKU_CSV,

        index=False,

        encoding="utf-8-sig"

    )


    maluku.to_excel(

        MALUKU_XLSX,

        index=False

    )


    # ========================================================
    # 9. SAVE MALUKU UTARA
    # ========================================================

    maluku_utara.to_csv(

        MALUKU_UTARA_CSV,

        index=False,

        encoding="utf-8-sig"

    )


    maluku_utara.to_excel(

        MALUKU_UTARA_XLSX,

        index=False

    )


    # ========================================================
    # 10. SUMMARY
    # ========================================================

    print()

    print("=" * 70)

    print(
        "SELESAI"
    )

    print("=" * 70)


    print()

    print(
        f"Total desa hasil        : {len(hasil)}"
    )

    print(
        f"Maluku                  : {len(maluku)}"
    )

    print(
        f"Maluku Utara            : {len(maluku_utara)}"
    )


    print()

    print(
        "STATISTIK JARAK"
    )

    print("-" * 70)


    print()

    print(
        "MALUKU"
    )

    print(
        f"Minimum : "
        f"{maluku['Jarak_KM'].min():.2f} km"
    )

    print(
        f"Rata-rata : "
        f"{maluku['Jarak_KM'].mean():.2f} km"
    )

    print(
        f"Median : "
        f"{maluku['Jarak_KM'].median():.2f} km"
    )

    print(
        f"Maksimum : "
        f"{maluku['Jarak_KM'].max():.2f} km"
    )


    print()

    print(
        "MALUKU UTARA"
    )

    print(
        f"Minimum : "
        f"{maluku_utara['Jarak_KM'].min():.2f} km"
    )

    print(
        f"Rata-rata : "
        f"{maluku_utara['Jarak_KM'].mean():.2f} km"
    )

    print(
        f"Median : "
        f"{maluku_utara['Jarak_KM'].median():.2f} km"
    )

    print(
        f"Maksimum : "
        f"{maluku_utara['Jarak_KM'].max():.2f} km"
    )


    print()

    print(
        "OUTPUT:"
    )

    print(
        MALUKU_CSV
    )

    print(
        MALUKU_XLSX
    )

    print(
        MALUKU_UTARA_CSV
    )

    print(
        MALUKU_UTARA_XLSX
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()